<a href="https://colab.research.google.com/github/gmauricio-toledo/NLP-LCC/blob/main/Tareas%20y%20Actividades/AC13%20-%20IR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Para obtener embeddings de documentos usando la arquitectura encoder-only transformers tenemos dos opciones:

1. Modelos de `sentence_transformers`
2. Mdelos de `transformers` cargados con `AutoModel`

En esta actividad usaremos la primera opción, y mostraremos una forma de hacerlo con la segunda.

In [ ]:
!gdown 1SXH1pwWV1-M32yyJ8PUYGi5W3N1G3oBz

In [ ]:
import pandas as pd

df = pd.read_csv('/content/all_top_rated_movies_from_TMDB.csv')
df = df[['original_title','overview']].copy()
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
docs = df['overview'].to_list()
df

In [ ]:
print(docs[25])

1. Revisa los textos imprimiendo aleatoriamente entradas (o usando cualquier otro método que prefieras). Encuentra 2 peliculas de referencia que te gustaría encontrar descripciones como queries.

2. Para cada una de las 2 películas, intenta encontrarla usando:

    * Una query que contenga algunas de las mismas palabras (aunque el texto no va a ser el mismo). Por ejemplo:

        Texto original: *After the death of her abusive husband, Matilde finds her new best friend in Miguel, her young, insecure, and disoriented neighbor.*

        Query: *Matilde is friends with her young neighbor.*

    * Una query que no contenga palabras iguales a la descripción, puedes resumir y parafrasear la descripción:

        Texto original: *After the death of her abusive husband, Matilde finds her new best friend in Miguel, her young, insecure, and disoriented neighbor.*

        Query: *A woman finds support in a younger man next door.*

3. Para cada una de las queries realiza la consulta de la siguiente forma:

    1. Vectoriza cada texto del corpus usando un modelo de `sentence_transformers`.
    2. Vectoriza la query usando el mismo modelo
    3. Usando la clase `sklearn.neighbors.NearestNeighbors` encuentra los $k$ vecinos más cercanos y verifica si encontraste la descripción de referencia que estabas buscando.

# Opción 1: Actividad en clase

🔴 Escoge un modelo de `sentence_transformers`

In [ ]:
from sentence_transformers import SentenceTransformer

model_id = ...
model = SentenceTransformer(model_id)

🔴 Obten los embeddings de documentos.

In [ ]:
embeddings = model.encode(docs)
embeddings.shape

🔴 Vectoriza las queries

In [ ]:
query = ...

query_vector = model.encode([query])

🔴 Inicializa la clase, eligen los hiperparámetros $k$ y métrica

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=,
                      metric=...
                      )
# Entrena la clase inicializada con los embeddings de los documentos


🟢 Encuentra los vecinos más cercanos a tu query

In [ ]:
vecinos = nn.kneighbors(query_vector)
print(vecinos)

🟢 Imprime el texto de los vecinos más cercanos

In [ ]:
distancias, idxs = vecinos
for idx in idxs[0]:
    print(docs[idx])
    print(50*"-")

🔴 Responde las siguientes preguntas

1. ¿Encontraste, con ambas queries cada uno de los dos textos de referencia? Rellena la información abajo

    |   | Documento de referencia 1  | Documento de referencia 1  |   
    |---|---|---|
    | Query textual  |   |   |
    | Query paráfrasis  |   |   |

    En cada caso escribe sí encontraste el texto de referencia y en qué ranking lo encontraste. **Prueba aumentar el parámetro k para encontrar el documento**


2. ¿Con cuál de las dos queries fué mejor?

3. ¿Qué puedes concluir sobre el desempeño de estos embeddings? Es decir, ¿qué información del texto captura mejor un modelo basado en transformadores? ¿la información textual o semántica?

# Opción 2: Sólo observar

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_id = "bert-base-uncased"
# model_id = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

In [ ]:
import torch

device = torch.device("cpu")  # Define el dispositivo como CPU
model.to(device)  # Mueve el modelo a la CPU

def get_cls_embedding(texts, verbose=False):
    # Tokeniza los textos: padding, truncamiento y conversión a tensores PyTorch
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=512)

    if verbose:
        # Calcula la longitud real de tokens (excluyendo padding)
        tokenized_length = len([idx for idx in encoded_input['input_ids'][0]
                               if idx != 0])
        print(f'Tokenized length: {tokenized_length}')

    # Mueve todos los inputs tokenizados al dispositivo (CPU/GPU)
    encoded_input = {key: value.to(device) for key, value in encoded_input.items()}

    # Desactiva el cálculo de gradientes para inferencia (más eficiente)
    with torch.no_grad():
        # Pasa los inputs por el modelo y obtiene todas las salidas
        model_output = model(**encoded_input, output_hidden_states=True)

    if verbose:
        # Muestra la forma del último layer de hidden states
        print(model_output['hidden_states'][-1].shape)

    # Extrae el embedding del token [CLS] (posición 0) del último layer
    cls_embeddings = model_output['hidden_states'][-1][:, 0, :]

    # Convierte a numpy array para uso fuera de PyTorch
    return cls_embeddings.numpy()

🟢 Obtenemos los embeddings

In [ ]:
import numpy as np

X = get_cls_embedding(docs, verbose=False)
X.shape

🟢 El resto del procedimiento es el mismo